In [ ]:
import numpy as np
import os
from tueplots import bundles
import matplotlib.pyplot as plt
import pickle
import scipy
## setting matplotlib context
from matplotlib.ticker import LogFormatterMathtext
from matplotlib.ticker import ScalarFormatter
from cycler import cycler
from matplotlib.cm import get_cmap
cmap = get_cmap("tab10",8)
palette = [cmap(i) for i in range(8)]
rc = bundles.neurips2024(usetex=False)
rc.update({
    # Set the line/bar color cycle (this is what affects ax.plot)
    "axes.prop_cycle": cycler(color=palette),
    # Optional readability tweaks
    "legend.frameon": False,
    "axes.grid": False,
})

In [ ]:
FPRS = [0.001,0.01,0.1]
M, N = 8192, 1000
COLOR_ALPHA=0.8
DATASET="adult_balanced"
path = f"./results/{DATASET}"
VERSION=2
with plt.rc_context(rc):
    fig, axes = plt.subplots(1,len(FPRS),sharex="row", layout="compressed")
    for f in range(len(FPRS)):
        FPR = FPRS[f]
        file = os.path.join(path, f"tabpfn_N_{N}_M_{M}_fpr_{FPR}.pkl")
        with open(file,"rb") as fl:
            res = pickle.load(fl)
        axes[f].plot(res["Ms"], res["TPR_concat"],color=palette[0],marker=".", linestyle=":", alpha = COLOR_ALPHA, label="Concatenated TPR (Naive)")
        axes[f].plot(res["Ms"], res["TPR_concat_pp"],color=palette[0], linestyle="-",marker=".", alpha = COLOR_ALPHA, label="Concatenated TPR (PP)")
        axes[f].plot(res["Ms"], res["TPR_concat_pp_tdist"],color=palette[1], linestyle="-",marker=".", alpha = COLOR_ALPHA, 
        label="Concatenated TPR (PP, Student's t)")
        axes[f].plot(res["Ms"], res["TPR_concat_pp_norm"],color=palette[2], linestyle="-", alpha = COLOR_ALPHA, 
        marker=".",label="Concatenated TPR (PP, Normal)")
        
        idx = len(res["TPR_px"])
        axes[f].plot(res["Ms"][len(res["Ms"])-idx:], res["TPR_px"], color=palette[3],marker=".", linestyle=":",alpha = COLOR_ALPHA, 
        label="Avg. TPR/Sample")
        axes[f].plot(res["Ms"][len(res["Ms"])-idx:], res["TPR_px_ana"],color=palette[3], alpha = COLOR_ALPHA, 
        linestyle="-",marker=".",label="Avg. TPR/Sample (Normal)")

        axes[f].set_yscale("log")
        axes[f].set_xscale("log", base=2)
        axes[f].set_title(f"FPR = {FPR}")
        axes[f].xaxis.set_major_formatter(ScalarFormatter())
        axes[f].ticklabel_format(axis="x", style="plain")
        
        # axes[f].xaxis.set_major_formatter(LogFormatterMathtext(base=2))
        # axes[f].xaxis.set_minor_formatter(LogFormatterMathtext(base=2))
        axes[f].set_box_aspect(1)
    
    axes[0].set_ylabel("TPR")
    axes[1].set_xlabel("M")

    h,l = axes[0].get_legend_handles_labels()
    fig.legend(h,l,
        loc="lower center",
        bbox_to_anchor=(0.55,0.1),
        ncol=3,
        frameon=False
    )
    plt.savefig(f"pp_tabpfn_{DATASET}_multi_fprs_v_{VERSION}.pdf",bbox_inches="tight")
    plt.show()


In [ ]:
BINS = 25
VERSION=1
with plt.rc_context(rc):
    fig, axes = plt.subplots(1,len(FPRS),layout="compressed")
    for f in range(len(FPRS)):
        FPR = FPRS[f]
        file = os.path.join(path, f"tabpfn_N_{N}_M_{M}_fpr_px_{FPR}.pkl")
        with open(file,"rb") as fl:
            res = pickle.load(fl)
        # print(res.keys())
        axes[f].hist(res["FPR_concat"],bins=BINS,alpha=0.5, density=True, label="Concatenated (Naive)")
        axes[f].hist(res["FPR_concat_pp"],bins=BINS,alpha=0.5, density=True, label="Concatenated (PP)")
        axes[f].hist(res["FPR_concat_pp_tdist"],bins=BINS,alpha=0.5, density=True, label="Concatenated (PP, Student's t)")
        axes[f].hist(res["FPR_concat_pp_norm"],bins=BINS,alpha=0.5, density=True, label="Concatenated (PP, Normal)")
        axes[f].axvline(x=FPR, color="r", alpha=COLOR_ALPHA,linestyle=":", label=f"Target FPR")
        axes[f].set_title(f"FPR = {FPR}")
        axes[f].set_box_aspect(1)
        
    axes[0].set_ylabel("count")
    axes[1].set_xlabel(rf"FPR$_x$")
    h,l = axes[0].get_legend_handles_labels()
    fig.legend(h,l,
        loc="lower center",
        bbox_to_anchor=(0.55,0.1),
        ncol=3,
        frameon=False
    )    
    plt.savefig(f"pp_tabpfn_{DATASET}_fprs_hist_v_{VERSION}.pdf",bbox_inches="tight")
    plt.show()


In [ ]:
with plt.rc_context(rc):
    fig, axes = plt.subplots(1,len(FPRS),sharex="row", layout="compressed")
    for f in range(len(FPRS)):
        FPR = FPRS[f]
        file = os.path.join(path, f"tabpfn_N_{N}_M_{M}_fpr_{FPR}_NOr.pkl")
        with open(file,"rb") as fl:
            res = pickle.load(fl)
        axes[f].plot(res["Ms"], res["TPR_concat"],color=palette[0],marker=".", linestyle=":", alpha = COLOR_ALPHA, label="Concatenated TPR (Naive)")
        axes[f].plot(res["Ms"], res["TPR_concat_pp"],color=palette[0], linestyle="-",marker=".", alpha = COLOR_ALPHA, label="Concatenated TPR (PP)")
        axes[f].plot(res["Ms"], res["TPR_concat_pp_norm"],color=palette[2], linestyle="-", alpha = COLOR_ALPHA, 
        marker=".",label="Concatenated TPR (PP, Normal)")
        axes[f].plot(res["Ms_TDIST"], res["TPR_concat_pp_tdist"],color=palette[1], linestyle="-",marker=".", alpha = COLOR_ALPHA, 
        label="Concatenated TPR (PP, Student's t)")
        # idx = len(res["TPR_px"])
        axes[f].plot(res["Ms_PX"], res["TPR_px"], color=palette[3],marker=".", linestyle=":",alpha = COLOR_ALPHA, 
        label="Avg. TPR/Sample")
        axes[f].plot(res["Ms_PX"], res["TPR_px_ana"],color=palette[3], alpha = COLOR_ALPHA, 
        linestyle="-",marker=".",label="Avg. TPR/Sample (Normal)")

        axes[f].set_yscale("log")
        axes[f].set_xscale("log", base=2)
        axes[f].set_title(f"FPR = {FPR}")
        axes[f].xaxis.set_major_formatter(ScalarFormatter())
        axes[f].ticklabel_format(axis="x", style="plain")
        if FPR != 0.1:
            axes[f].set_ylim(bottom=FPR)
        # axes[f].xaxis.set_major_formatter(LogFormatterMathtext(base=2))
        # axes[f].xaxis.set_minor_formatter(LogFormatterMathtext(base=2))
        axes[f].set_box_aspect(1)
    
    axes[0].set_ylabel("TPR")
    axes[1].set_xlabel("M")

    h,l = axes[0].get_legend_handles_labels()
    fig.legend(h,l,
        loc="lower center",
        bbox_to_anchor=(0.55,0.1),
        ncol=3,
        frameon=False
    )
    plt.savefig(f"pp_tabpfn_{DATASET}_multi_fprs_v_{VERSION}_NOr.pdf",bbox_inches="tight")
    plt.show()
